In [1]:
import operator
from googleapiclient.discovery import build
from dotenv import load_dotenv
import os
import re

In [2]:
load_dotenv(override=True)
API_KEY = os.getenv("YOUTUBE_API_KEY")

In [3]:

youtube = build("youtube", "v3", developerKey=API_KEY)
video_ids = ['Gvt4TwGpqOs',
 'Jj1-zb38Yfw',
 'Jj1-zb38Yfw',
 'tr5Fapv80Cw',
 'MrD9tCNpOvU',
 '-pqzyvRp3Tc']


In [4]:
request = youtube.videos().list(
            part="snippet,statistics,contentDetails",
            id=",".join(video_ids)
            )

response = request.execute()
print(response)

TimeoutError: [WinError 10060] A connection attempt failed because the connected party did not properly respond after a period of time, or established connection failed because connected host has failed to respond

In [17]:
def get_counts(video_ids):
    request = youtube.videos().list(
            part="snippet,statistics,contentDetails",
            id=",".join(video_ids)
            )

    response = request.execute()
    video_summaries = {}
    for video in response["items"]:
        views = int(video["statistics"]["viewCount"])
        likes = int(video["statistics"]["likeCount"])
        comments = int(video["statistics"]["commentCount"])
        video_summaries[video["id"]] = {"views" : views, "likes" : likes, "comments" : comments, "title": video["snippet"]["title"], "channelTitle" : video["snippet"]["channelTitle"]}
    return video_summaries



In [22]:
video_ids_dict = get_counts(video_ids)
video_ids_dict

{'Gvt4TwGpqOs': {'views': 82706,
  'likes': 8549,
  'comments': 1020,
  'title': 'Explaining Agentic AI: The Good, the Bad & the Ugly',
  'channelTitle': 'ExplainingComputers'},
 'Jj1-zb38Yfw': {'views': 136681,
  'likes': 3283,
  'comments': 66,
  'title': 'Agentic AI Explained So Anyone Can Get It!',
  'channelTitle': 'ByteMonk'},
 'tr5Fapv80Cw': {'views': 62390,
  'likes': 1795,
  'comments': 42,
  'title': 'Building Agentic AI Workloads – Crash Course',
  'channelTitle': 'freeCodeCamp.org'},
 'MrD9tCNpOvU': {'views': 59938,
  'likes': 852,
  'comments': 22,
  'title': 'Agentic AI Design Patterns Introduction and walkthrough | Amazon Web Services',
  'channelTitle': 'Amazon Web Services'},
 '-pqzyvRp3Tc': {'views': 199256,
  'likes': 2448,
  'comments': 57,
  'title': 'What is Agentic AI? An Easy Explanation For Everyone',
  'channelTitle': 'Bernard Marr'}}

In [31]:
def get_authority_scores(video_ids_dict):
    max_views= max(v['views'] for v in video_ids_dict.values())
    min_views= min(v['views'] for v in video_ids_dict.values())
    max_likes= max(v['likes'] for v in video_ids_dict.values())
    min_likes= min(v['likes'] for v in video_ids_dict.values())
    max_comments= max(v['comments'] for v in video_ids_dict.values())
    min_comments= min(v['comments'] for v in video_ids_dict.values())
    authority_scores_dict = video_ids_dict.copy()
    for video_id in video_ids_dict.keys():
        current_views = video_ids_dict[video_id]['views']
        current_likes = video_ids_dict[video_id]['likes']
        current_comments = video_ids_dict[video_id]['comments']
        normalized_views = (current_views - min_views) / (max_views - min_views)
        normalized_likes = (current_likes - min_likes) / (max_likes - min_likes)
        normalized_comments = (current_comments - min_comments) / (max_comments - min_comments)
        authority_score = 0.5 * normalized_views + 0.3 * normalized_likes + 0.2 * normalized_comments
        authority_scores_dict[video_id]["authority_score"] = round(authority_score,2)
    return authority_scores_dict



In [32]:

authority_scores_dict = get_authority_scores(video_ids_dict)
print(authority_scores_dict)





{'Gvt4TwGpqOs': {'views': 82706, 'likes': 8549, 'comments': 1020, 'title': 'Explaining Agentic AI: The Good, the Bad & the Ugly', 'channelTitle': 'ExplainingComputers', 'authority_score': 0.58}, 'Jj1-zb38Yfw': {'views': 136681, 'likes': 3283, 'comments': 66, 'title': 'Agentic AI Explained So Anyone Can Get It!', 'channelTitle': 'ByteMonk', 'authority_score': 0.38}, 'tr5Fapv80Cw': {'views': 62390, 'likes': 1795, 'comments': 42, 'title': 'Building Agentic AI Workloads – Crash Course', 'channelTitle': 'freeCodeCamp.org', 'authority_score': 0.05}, 'MrD9tCNpOvU': {'views': 59938, 'likes': 852, 'comments': 22, 'title': 'Agentic AI Design Patterns Introduction and walkthrough | Amazon Web Services', 'channelTitle': 'Amazon Web Services', 'authority_score': 0.0}, '-pqzyvRp3Tc': {'views': 199256, 'likes': 2448, 'comments': 57, 'title': 'What is Agentic AI? An Easy Explanation For Everyone', 'channelTitle': 'Bernard Marr', 'authority_score': 0.57}}


In [33]:
def get_video_summaries(authority_scores_dict):
    video_summaries = []
    for video_id in authority_scores_dict.keys():
        score = authority_scores_dict[video_id]["authority_score"]
        if score >= 0.8:
            recommendation = "recommended as core curriculum material"
        elif score >= 0.6:
            recommendation = "recommended as primary supporting material"
        elif score >= 0.4:
            recommendation = "suitable as supplementary learning"
        else:
            recommendation = "optional reference material"
        text = f"""
        The video "{video_ids_dict[video_id]['title']}" published by
        {video_ids_dict[video_id]['channelTitle']}
        achieves an authority score of {score:.2f}.

        Based on engagement and quality metrics, it is {recommendation}
        for learners exploring this topic.
        """
        video_summaries.append(text)
    return video_summaries



In [34]:
print([re.sub(r"\n|\s{2,}", " ", video_summary).strip() for video_summary in get_video_summaries(authority_scores_dict)])

['The video "Explaining Agentic AI: The Good, the Bad & the Ugly" published by  ExplainingComputers  achieves an authority score of 0.58.   Based on engagement and quality metrics, it is suitable as supplementary learning  for learners exploring this topic.', 'The video "Agentic AI Explained So Anyone Can Get It!" published by  ByteMonk  achieves an authority score of 0.38.   Based on engagement and quality metrics, it is optional reference material  for learners exploring this topic.', 'The video "Building Agentic AI Workloads – Crash Course" published by  freeCodeCamp.org  achieves an authority score of 0.05.   Based on engagement and quality metrics, it is optional reference material  for learners exploring this topic.', 'The video "Agentic AI Design Patterns Introduction and walkthrough | Amazon Web Services" published by  Amazon Web Services  achieves an authority score of 0.00.   Based on engagement and quality metrics, it is optional reference material  for learners exploring th